In [1]:
library(dplyr)
library(stringr)
library(vroom)
library(tidyr)

Warning message:
“package ‘dplyr’ was built under R version 4.3.2”

Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Warning message:
“package ‘stringr’ was built under R version 4.3.2”


In [2]:
vcf.df <- data.table::fread('/nfs/lab/tscc/welison/FNIH.Liver/genotypes/241009_WE_class_matrix.rsID.tsv')
vcf.df <- tibble::column_to_rownames(vcf.df, 'V1')


#rownames(vcf.df) <- NULL
#vcf.df <- tibble::column_to_rownames(vcf.df, 'ID')

dim(vcf.df)
head(vcf.df)

Warning message in data.table::fread("/nfs/lab/tscc/welison/FNIH.Liver/genotypes/241009_WE_class_matrix.rsID.tsv"):
“Detected 87 column names but the data has 88 columns (i.e. invalid file). Added 1 extra default column name for the first column which is guessed to be row names or an index. Use setnames() afterwards if this guess is not correct, or fix the file write command that created the file to create a valid file.”


[1] 5492117      87

,ID,HL170044,HL200529,HL230212,HL220405,HL160017,HL170064,HL220402,HL150015,HL181029,⋯,HL170066,HL180071,HL170060,HL211122,HL221215,HL210806,HL190606,HL191216,HL210120,HL181010
,<chr>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,⋯,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
chr10:116162:A:C,rs11594819,1,1,0,1,0,0,1,1,1,⋯,1,1,2,1,0,2,1,2,2,0
chr10:125824:C:G,rs72770958,1,0,0,0,0,0,1,0,0,⋯,0,0,0,0,0,0,0,0,0,0
chr10:132494:T:G,rs7089889,2,1,2,1,2,2,2,2,1,⋯,0,1,1,1,2,2,1,1,1,2
chr10:136160:C:T,rs73581705,1,0,0,1,0,0,1,1,1,⋯,0,1,1,0,1,0,0,0,0,1
chr10:142865:A:G,rs12146291,1,0,0,1,0,0,1,1,1,⋯,0,1,1,0,1,0,0,0,0,1
chr10:155638:G:A,rs17156310,1,0,0,1,0,0,1,1,1,⋯,0,1,1,0,1,0,0,0,0,1


In [6]:
cred.set <- read.table('/nfs/lab/projects/nash_nafld_liver/GWAS/MVP_GWAS/NAFLD.TRANS.MVP.2021.credset.hg38.tsv', header=T, sep='\t')

dim(cred.set)
head(cred.set)

[1] 1084   18

,CS.Type,LEAD.SNP,CS.SNP,Chr,Position,EA,NEA,EAF,Beta,SE,P,N,PP,Nominated.Gene,Biological.Prior.Gene,chr.hg38,end.hg38,start.hg38
,<chr>,<chr>,<chr>,<int>,<int>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<chr>,<chr>,<chr>,<int>,<int>,<int>
1,Single SNP,rs2642438,rs2642438,1,220970028,A,G,0.274,-0.075,0.0074,6.65e-24,218595,96.86%,MTARC1,MTARC1,1,220796686,220796685
2,,rs6734238,rs6734238,2,113841030,G,A,0.407,-0.057,0.0064,4.94e-19,218595,99.04%,IL1RN,IL1RN,2,113083453,113083452
3,,rs138033684,rs138033684,6,71895252,G,T,0.006,0.677,0.1120,1.42e-09,37364,96.31%,OGFRL1,OGFRL1,6,71185549,71185548
4,,rs2980888,rs2980888,8,126507308,T,C,0.286,0.130,0.0072,4.21e-72,218595,99.95%,lnc-TRIB1-2;WASHC5,TRIB1,8,125495066,125495065
5,,rs4484649,rs4484649,8,10571491,C,A,0.419,0.045,0.0066,1.38e-11,218595,98.51%,SOX7;RP1L1;C8orf74,RP1L1;SOX7,8,10713981,10713980
6,,rs4841132,rs4841132,8,9183596,A,G,0.106,0.123,0.0105,6.62e-32,218595,98.79%,PPP1R3B,PPP1R3B;TNKS;MFHAS1,8,9326086,9326085


In [39]:
head(cred.set$CS.SNP)

[1] "rs2642438"   "rs6734238"   "rs138033684" "rs2980888"   "rs4484649"  
[6] "rs4841132"

In [40]:
sum(vcf.df$ID %in% cred.set$CS.SNP)

cs.variants <- cred.set$CS.SNP[cred.set$CS.SNP %in% vcf.df$ID]
length(cs.variants)
head(cs.variants)

[1] 922

[1] 922

[1] "rs2642438" "rs6734238" "rs2980888" "rs4484649" "rs4841132" "rs738408"

In [8]:
cred.set[!cred.set$CS.SNP %in% vcf.df$ID,]

,CS.Type,LEAD.SNP,CS.SNP,Chr,Position,EA,NEA,EAF,Beta,SE,P,N,PP,Nominated.Gene,Biological.Prior.Gene,chr.hg38,end.hg38,start.hg38
,<chr>,<chr>,<chr>,<int>,<int>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<int>,<chr>,<chr>,<chr>,<int>,<int>,<int>
3,,rs138033684,rs138033684,6,71895252,G,T,0.006,0.677,0.1120,1.42e-09,37364,96.31%,OGFRL1,OGFRL1,6,71185549,71185548
10,,rs4782568,rs4782568,16,83980529,G,C,0.420,-0.064,0.0068,8.82e-21,218595,97.91%,OSGIN1;MLYCD,OSGIN1;MLYCD,16,83946924,83946923
16,,rs1337101,rs78190323,1,219719625,G,A,0.327,-0.046,0.0069,4.59e-11,218595,3.16%,SLC30A10,LYPLAL1;SLC30A10,1,219546283,219546282
61,,rs6541349,rs35910103,1,93839030,G,A,0.198,0.045,0.0082,3.67e-08,218595,0.97%,CCDC18,CCDC18;FNBP1L,1,93373473,93373472
66,,rs6541349,rs59955169,1,93768768,T,C,0.289,0.044,0.0081,6.34e-08,218595,0.71%,CCDC18,CCDC18;FNBP1L,1,93303211,93303210
68,,rs6541349,rs6541379,1,93896765,A,G,0.113,0.059,0.0109,7.45e-08,218595,0.69%,CCDC18,CCDC18;FNBP1L,1,93431208,93431207
82,,rs6541349,rs2162277,1,93900247,C,T,0.152,0.051,0.0096,1.30e-07,218595,0.33%,CCDC18,CCDC18;FNBP1L,1,93434690,93434689
83,,rs74816838,rs74816838,1,161643560,T,C,0.106,0.078,0.0122,1.51e-10,218595,87.03%,FCGR2A;FCGR2B,FCGR2A;FCGR2B,1,161673770,161673769
84,,rs74816838,rs61804205,1,161653737,C,T,0.099,0.065,0.0112,9.64e-09,218595,1.83%,FCGR2A;FCGR2B,FCGR2A;FCGR2B,1,161683947,161683946


In [33]:
vcf.df[str_detect(rownames(vcf.df), 'chr17:66232'),]

,ID,HL170044,HL200529,HL230212,HL220405,HL160017,HL170064,HL220402,HL150015,HL181029,⋯,HL170066,HL180071,HL170060,HL211122,HL221215,HL210806,HL190606,HL191216,HL210120,HL181010
,<chr>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,⋯,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>
chr17:66232900:C:T,rs8073149,1,0,0,2,0,1,0,0,1,⋯,0,0,1,0,1,0,0,0,0,0


In [70]:
cs.qtl.sumstats <- data.frame()

for (cell in c('B','Cholangiocyte','Endothelial','Hepatocytes','HSC','Myeloid','NK','T')) {
    qtl.sumstats <- vroom(paste0('/nfs/lab/tscc/welison/FNIH.Liver/tensorQTL/04_eQTLs/outs/',cell,'.cis_qtl_pairs.chr.qvalue.sig.tsv'), delim = '\t')
    cs.qtl.sumstats <- filter(qtl.sumstats, variant_id %in% cs.variants) %>%
        mutate(cell=cell, modality='RNA') %>%
        rbind(cs.qtl.sumstats)
}

dim(cs.qtl.sumstats)
head(cs.qtl.sumstats)

Rows: 653 Columns: 11
── Column specification ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: "\t"
chr (2): phenotype_id, variant_id
dbl (8): start_distance, af, ma_samples, ma_count, pval_nominal, slope, slop...
lgl (1): sig.qtl

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 8140 Columns: 11
── Column specification ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: "\t"
chr (2): phenotype_id, variant_id
dbl (8): start_distance, af, ma_samples, ma_count, pval_nominal, slope, slop...
lgl (1): sig.qtl

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 2

[1] 52 13

phenotype_id,variant_id,start_distance,af,ma_samples,ma_count,pval_nominal,slope,slope_se,sig.qtl,pval_nominal_threshold,cell,modality
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<lgl>,<dbl>,<chr>,<chr>
HSD17B13,rs10433879,-12898,0.2034884,30,35,4.227959e-06,0.7136446,0.13917176,TRUE,1.03562e-05,Myeloid,RNA
HLA-DQB1,rs968155,-255445,0.6453488,48,61,8.382722e-06,-0.5900655,0.11959907,TRUE,1.02602e-05,Myeloid,RNA
HLA-DQB1,rs9271406,-48572,0.5930232,54,70,8.116191e-08,-0.6510572,0.10469777,TRUE,1.02602e-05,Myeloid,RNA
EPHA2,rs11588341,15484,0.5232558,61,82,9.513265e-06,-0.3773269,0.07704365,TRUE,2.53574e-05,Hepatocytes,RNA
EPHA2,rs1497407,17041,0.4883721,60,84,8.349330e-06,-0.3804022,0.07708509,TRUE,2.53574e-05,Hepatocytes,RNA
EPHA2,rs7519043,17959,0.5174419,62,83,6.773970e-06,-0.3836847,0.07682361,TRUE,2.53574e-05,Hepatocytes,RNA


In [71]:
#cs.qtl.sumstats <- data.frame()

for (cell in c('B','Cholangiocyte','Endothelial','Hepatocytes','HSC','Myeloid','NK','T')) {
    qtl.sumstats <- vroom(paste0('/nfs/lab/tscc/welison/FNIH.Liver/tensorQTL/06_H3K27acQTLs/outs/',cell,'.cis_qtl_pairs.chr.qvalue.sig.tsv'), delim = '\t')
    cs.qtl.sumstats <- filter(qtl.sumstats, variant_id %in% cs.variants) %>%
        select(-end_distance) %>%
        mutate(cell=cell, modality='H3K27ac') %>%
        rbind(cs.qtl.sumstats)
}

dim(cs.qtl.sumstats)
head(cs.qtl.sumstats)

Rows: 0 Columns: 11
── Column specification ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: "\t"
chr (11): phenotype_id, variant_id, start_distance, end_distance, af, ma_sam...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 0 Columns: 11
── Column specification ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: "\t"
chr (11): phenotype_id, variant_id, start_distance, end_distance, af, ma_sam...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 1653 Columns: 12
── Column specification ───────────────────────────────────────────────────────────────────

[1] 153  13

phenotype_id,variant_id,start_distance,af,ma_samples,ma_count,pval_nominal,slope,slope_se,sig.qtl,pval_nominal_threshold,cell,modality
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<lgl>,<dbl>,<chr>,<chr>
chr8:8227666-8228665,rs2979172,225331,0.6395349,47,62,1.630880e-09,-0.9632696,0.1324173,TRUE,1.09088e-07,Myeloid,H3K27ac
chr8:8227666-8228665,rs4841040,569350,0.5872093,56,71,5.232810e-09,-0.9810939,0.1409638,TRUE,1.09088e-07,Myeloid,H3K27ac
chr8:8227666-8228665,rs6994038,575361,0.5872093,56,71,2.781884e-08,-0.9362662,0.1438492,TRUE,1.09088e-07,Myeloid,H3K27ac
chr8:8227666-8228665,rs4841042,579445,0.5930232,54,70,5.722191e-09,-0.9160417,0.1320748,TRUE,1.09088e-07,Myeloid,H3K27ac
chr8:8227666-8228665,rs7823757,585000,0.5813953,56,72,6.893619e-09,-0.9762346,0.1417808,TRUE,1.09088e-07,Myeloid,H3K27ac
chr8:8227666-8228665,rs60315134,585422,0.5813953,56,72,6.893619e-09,-0.9762346,0.1417808,TRUE,1.09088e-07,Myeloid,H3K27ac


In [72]:
#cs.qtl.sumstats <- data.frame()

for (cell in c('B','Cholangiocyte','Endothelial','Hepatocytes','HSC','Myeloid','NK','T')) {
    qtl.sumstats <- vroom(paste0('/nfs/lab/tscc/welison/FNIH.Liver/tensorQTL/07_H3K27me3QTLs/outs/',cell,'.cis_qtl_pairs.chr.qvalue.sig.tsv'), delim = '\t')
    cs.qtl.sumstats <- filter(qtl.sumstats, variant_id %in% cs.variants) %>%
        select(-end_distance) %>%
        mutate(cell=cell, modality='H3K27me3') %>%
        rbind(cs.qtl.sumstats)
}

dim(cs.qtl.sumstats)
head(cs.qtl.sumstats)

Rows: 0 Columns: 11
── Column specification ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: "\t"
chr (11): phenotype_id, variant_id, start_distance, end_distance, af, ma_sam...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 54 Columns: 12
── Column specification ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: "\t"
chr (2): phenotype_id, variant_id
dbl (9): start_distance, end_distance, af, ma_samples, ma_count, pval_nomina...
lgl (1): sig.qtl

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 0 Columns: 11
── Column specification ──────────────────

[1] 167  13

phenotype_id,variant_id,start_distance,af,ma_samples,ma_count,pval_nominal,slope,slope_se,sig.qtl,pval_nominal_threshold,cell,modality
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<lgl>,<dbl>,<chr>,<chr>
chr6:32550868-32552367,rs9268839,-89874,0.5174419,64,83,1.36998e-07,0.7758834,0.1276958,TRUE,4.81518e-07,Hepatocytes,H3K27me3
chr6:32550868-32552367,rs13211921,-75810,0.5174419,64,83,1.36998e-07,0.7758834,0.1276958,TRUE,4.81518e-07,Hepatocytes,H3K27me3
chr6:32550868-32552367,rs9391879,-75474,0.5174419,64,83,1.36998e-07,0.7758834,0.1276958,TRUE,4.81518e-07,Hepatocytes,H3K27me3
chr6:32550868-32552367,rs12195589,-74062,0.5174419,64,83,1.36998e-07,0.7758834,0.1276958,TRUE,4.81518e-07,Hepatocytes,H3K27me3
chr6:32550868-32552367,rs28895253,-73736,0.5174419,64,83,1.36998e-07,0.7758834,0.1276958,TRUE,4.81518e-07,Hepatocytes,H3K27me3
chr6:32550868-32552367,rs28895257,-73532,0.5174419,64,83,1.36998e-07,0.7758834,0.1276958,TRUE,4.81518e-07,Hepatocytes,H3K27me3


In [73]:
#cs.qtl.sumstats <- data.frame()

for (cell in c('B','Cholangiocyte','Endothelial','Hepatocytes','HSC','Myeloid','NK','T')) {
    for (i in 1:22) {
        qtl.sumstats <- vroom(paste0('/nfs/lab/tscc/welison/FNIH.Liver/tensorQTL/05_caQTLs/outs/',cell,'.cis_qtl_pairs.chr',i,'.parquet.sig.tsv'), delim = '\t')
        cs.qtl.sumstats <- filter(qtl.sumstats, variant_id %in% cs.variants) %>%
            select(-end_distance) %>%
            mutate(cell=cell, modality='ATAC') %>%
            rbind(cs.qtl.sumstats)
    }
}

dim(cs.qtl.sumstats)
head(cs.qtl.sumstats)

Rows: 0 Columns: 12
── Column specification ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: "\t"
chr (12): phenotype_id, variant_id, start_distance, end_distance, af, ma_sam...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 0 Columns: 12
── Column specification ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: "\t"
chr (12): phenotype_id, variant_id, start_distance, end_distance, af, ma_sam...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 6 Columns: 12
── Column specification ──────────────────────────────────────────────────────────────────────


ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 0 Columns: 12
── Column specification ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: "\t"
chr (12): phenotype_id, variant_id, start_distance, end_distance, af, ma_sam...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 3 Columns: 12
── Column specification ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: "\t"
chr (2): phenotype_id, variant_id
dbl (9): start_distance, end_distance, af, ma_samples, ma_count, pval_nomina...
lgl (1): sig.qtl

ℹ Use `spec()` to retrieve the full column specification for t


ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 747 Columns: 12
── Column specification ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: "\t"
chr (2): phenotype_id, variant_id
dbl (9): start_distance, end_distance, af, ma_samples, ma_count, pval_nomina...
lgl (1): sig.qtl

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 10 Columns: 12
── Column specification ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: "\t"
chr (2): phenotype_id, variant_id
dbl (9): start_distance, end_distance, af, ma_samples, ma_count, pval_nomina...
lgl (1): sig.qtl

ℹ Use `s

── Column specification ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: "\t"
chr (2): phenotype_id, variant_id
dbl (9): start_distance, end_distance, af, ma_samples, ma_count, pval_nomina...
lgl (1): sig.qtl

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 39 Columns: 12
── Column specification ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: "\t"
chr (2): phenotype_id, variant_id
dbl (9): start_distance, end_distance, af, ma_samples, ma_count, pval_nomina...
lgl (1): sig.qtl

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 6318 Columns: 12
── Colum


ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 1333 Columns: 12
── Column specification ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: "\t"
chr (2): phenotype_id, variant_id
dbl (9): start_distance, end_distance, af, ma_samples, ma_count, pval_nomina...
lgl (1): sig.qtl

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 4407 Columns: 12
── Column specification ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: "\t"
chr (2): phenotype_id, variant_id
dbl (9): start_distance, end_distance, af, ma_samples, ma_count, pval_nomina...
lgl (1): sig.qtl

ℹ Use

...


ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 299 Columns: 12
── Column specification ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: "\t"
chr (2): phenotype_id, variant_id
dbl (9): start_distance, end_distance, af, ma_samples, ma_count, pval_nomina...
lgl (1): sig.qtl

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 28 Columns: 12
── Column specification ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: "\t"
chr (2): phenotype_id, variant_id
dbl (9): start_distance, end_distance, af, ma_samples, ma_count, pval_nomina...
lgl (1): sig.qtl

ℹ Use `s

── Column specification ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: "\t"
chr (2): phenotype_id, variant_id
dbl (9): start_distance, end_distance, af, ma_samples, ma_count, pval_nomina...
lgl (1): sig.qtl

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 0 Columns: 12
── Column specification ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: "\t"
chr (12): phenotype_id, variant_id, start_distance, end_distance, af, ma_sam...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 90 Columns: 12
── Column specification ──────────────────────────────────────


ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 34 Columns: 12
── Column specification ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: "\t"
chr (2): phenotype_id, variant_id
dbl (9): start_distance, end_distance, af, ma_samples, ma_count, pval_nomina...
lgl (1): sig.qtl

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 215 Columns: 12
── Column specification ───────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: "\t"
chr (2): phenotype_id, variant_id
dbl (9): start_distance, end_distance, af, ma_samples, ma_count, pval_nomina...
lgl (1): sig.qtl

ℹ Use `s

[1] 1472   13

phenotype_id,variant_id,start_distance,af,ma_samples,ma_count,pval_nominal,slope,slope_se,pval_nominal_threshold,sig.qtl,cell,modality
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<lgl>,<chr>,<chr>
chr8-8227864-8228164,rs2979172,225133,0.6395349,47,62,5.864511e-09,-0.9794559,0.1413530,2.04174e-06,TRUE,T,ATAC
chr8-8227864-8228164,rs4841040,569152,0.5872093,56,71,5.056431e-09,-1.0468580,0.1502131,2.04174e-06,TRUE,T,ATAC
chr8-8227864-8228164,rs6994038,575163,0.5872093,56,71,7.260357e-10,-1.0916997,0.1456938,2.04174e-06,TRUE,T,ATAC
chr8-8227864-8228164,rs4841042,579247,0.5930232,54,70,1.310349e-09,-1.0138221,0.1382438,2.04174e-06,TRUE,T,ATAC
chr8-8227864-8228164,rs7823757,584802,0.5813953,56,72,6.011590e-09,-1.0352899,0.1495551,2.04174e-06,TRUE,T,ATAC
chr8-8227864-8228164,rs60315134,585224,0.5813953,56,72,6.011590e-09,-1.0352899,0.1495551,2.04174e-06,TRUE,T,ATAC


In [61]:
cred.set.to.join <- cred.set %>%
    select(-CS.Type, -Chr, -Position, chr=chr.hg38, -start.hg38, position=end.hg38, EA, NEA, -EAF, Beta.GWAS=Beta, SE.GWAS=SE, P.GWAS=P,
          N.GWAS=N, PP, -Nominated.Gene, -Biological.Prior.Gene) %>%
    relocate(LEAD.SNP, CS.SNP, chr, position, EA, NEA, Beta.GWAS, SE.GWAS, P.GWAS, N.GWAS, PP)

dim(cred.set.to.join)
head(cred.set.to.join)

[1] 1084   11

,LEAD.SNP,CS.SNP,chr,position,EA,NEA,Beta.GWAS,SE.GWAS,P.GWAS,N.GWAS,PP
,<chr>,<chr>,<int>,<int>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<int>,<chr>
1,rs2642438,rs2642438,1,220796686,A,G,-0.075,0.0074,6.65e-24,218595,96.86%
2,rs6734238,rs6734238,2,113083453,G,A,-0.057,0.0064,4.94e-19,218595,99.04%
3,rs138033684,rs138033684,6,71185549,G,T,0.677,0.1120,1.42e-09,37364,96.31%
4,rs2980888,rs2980888,8,125495066,T,C,0.130,0.0072,4.21e-72,218595,99.95%
5,rs4484649,rs4484649,8,10713981,C,A,0.045,0.0066,1.38e-11,218595,98.51%
6,rs4841132,rs4841132,8,9326086,A,G,0.123,0.0105,6.62e-32,218595,98.79%


In [74]:
cs.qtl.sumstats.joined <- cs.qtl.sumstats %>%
    left_join(cred.set.to.join, by=join_by(variant_id==CS.SNP))

dim(cs.qtl.sumstats.joined)
head(cs.qtl.sumstats.joined)

[1] 1472   23

phenotype_id,variant_id,start_distance,af,ma_samples,ma_count,pval_nominal,slope,slope_se,pval_nominal_threshold,⋯,LEAD.SNP,chr,position,EA,NEA,Beta.GWAS,SE.GWAS,P.GWAS,N.GWAS,PP
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,⋯,<chr>,<int>,<int>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<int>,<chr>
chr8-8227864-8228164,rs2979172,225133,0.6395349,47,62,5.864511e-09,-0.9794559,0.1413530,2.04174e-06,⋯,rs60315134,8,8452998,C,G,0.034,0.0065,2.41e-07,218595,0.29%
chr8-8227864-8228164,rs4841040,569152,0.5872093,56,71,5.056431e-09,-1.0468580,0.1502131,2.04174e-06,⋯,rs60315134,8,8797017,C,T,0.034,0.0066,3.76e-07,218595,0.20%
chr8-8227864-8228164,rs6994038,575163,0.5872093,56,71,7.260357e-10,-1.0916997,0.1456938,2.04174e-06,⋯,rs60315134,8,8803028,A,C,0.034,0.0065,2.63e-07,218595,0.29%
chr8-8227864-8228164,rs4841042,579247,0.5930232,54,70,1.310349e-09,-1.0138221,0.1382438,2.04174e-06,⋯,rs60315134,8,8807112,A,G,0.034,0.0065,1.41e-07,218595,0.47%
chr8-8227864-8228164,rs7823757,584802,0.5813953,56,72,6.011590e-09,-1.0352899,0.1495551,2.04174e-06,⋯,rs60315134,8,8812667,A,T,0.035,0.0066,6.40e-08,218595,0.81%
chr8-8227864-8228164,rs60315134,585224,0.5813953,56,72,6.011590e-09,-1.0352899,0.1495551,2.04174e-06,⋯,rs60315134,8,8813089,G,A,0.036,0.0065,2.88e-08,218595,2.68%


In [75]:
cs.qtl.sumstats.joined %>%
    group_by(LEAD.SNP, modality) %>%
    summarise(n())

`summarise()` has grouped output by 'LEAD.SNP'. You can override using the `.groups` argument.


LEAD.SNP,modality,n()
<chr>,<chr>,<int>
rs10433937,RNA,13
rs10883451,RNA,1
rs11621792,ATAC,2
rs11621792,H3K27ac,1
rs11683367,ATAC,61
rs11683367,H3K27ac,13
rs11683367,RNA,3
rs11683409,ATAC,88
rs12149380,ATAC,2


In [82]:
length(unique(cred.set.to.join$LEAD.SNP))

[1] 77

In [76]:
cs.qtl.sumstats.joined %>%
    select(LEAD.SNP, modality) %>%
    table()

            modality
LEAD.SNP     ATAC H3K27ac H3K27me3 RNA
  rs10433937    0       0        0  13
  rs10883451    0       0        0   1
  rs11621792    2       1        0   0
  rs11683367   61      13        0   3
  rs11683409   88       0        0   0
  rs12149380    2       0        0   0
  rs1547014    18       0        0   4
  rs1626329    32       0        0   0
  rs174535      5       0        0   0
  rs2727324    17       0        0   0
  rs2737217     4       0        0   0
  rs2943652    32       0        0   0
  rs34123446  134       0        0   0
  rs35199395    4       0        0   0
  rs36086195   46       0        0  10
  rs3810367    41       9        0   0
  rs3935942     2       0        0   0
  rs4484649     2       0        0   0
  rs4683438     2       0        0   0
  rs4805033     4       0        0   0
  rs4841132     2       0        0   2
  rs4918722    30       0        0   0
  rs56094641   13       0        0   0
  rs56175344    4       0        0   0
  rs

In [83]:
cs.qtl.sumstats.joined %>%
    select(LEAD.SNP, cell) %>%
    table()

            cell
LEAD.SNP       B Cholangiocyte Endothelial Hepatocytes HSC Myeloid  NK   T
  rs10433937   0             0           0          12   0       1   0   0
  rs10883451   0             0           0           1   0       0   0   0
  rs11621792   0             0           0           3   0       0   0   0
  rs11683367   0             0           0          77   0       0   0   0
  rs11683409   0             0          44          44   0       0   0   0
  rs12149380   0             0           2           0   0       0   0   0
  rs1547014    0             0           0          22   0       0   0   0
  rs1626329    0             0           0           0   0      32   0   0
  rs174535     0             0           0           5   0       0   0   0
  rs2727324    0             0           0           0   0      17   0   0
  rs2737217    0             0           0           0   0       4   0   0
  rs2943652    0             0           0          18   0      14   0   0
  rs3412

In [78]:
write.table(cs.qtl.sumstats.joined, '/nfs/lab/tscc/welison/FNIH.Liver/tensorQTL/NAFLD.TRANS.MVP.2021.credset.hg38.QTL.variant.intersect.tsv',
           sep='\t', col.names=T, row.names=F, quote=F)